<a href="https://colab.research.google.com/github/allvaret/yolo-traffic-pipeline/blob/main/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# yolo-traffic-pipeline — Colab runner

Esse notebook só orquestra o repositório `yolo-traffic-pipeline`; a lógica de verdade
mora nos módulos `.py` (não em células soltas), pra ficar reaproveitável fora do Colab
também.

Antes de rodar: Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4).

## 1. Clonar o repositório


In [1]:
!git clone https://github.com/allvaret/yolo-traffic-pipeline.git
%cd yolo-traffic-pipeline
!pip install -q -r requirements.txt


Cloning into 'yolo-traffic-pipeline'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 54 (delta 17), reused 45 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 26.79 KiB | 5.36 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/yolo-traffic-pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.6/18.6 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━

## 2. Configurar a API key do Roboflow

Gere uma chave gratuita em roboflow.com/settings/api, depois rode a célula abaixo
e cole quando pedir (fica só na sessão, não é salva em lugar nenhum).

In [2]:
import os
from getpass import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass("Cole sua Roboflow API key: ")


Cole sua Roboflow API key: ··········


## 3. Preencher config/sources.yaml

Antes de rodar a etapa de dataset, confira se `config/sources.yaml` já tem os
`workspace`/`project`/`version` do Roboflow preenchidos pra
pedestrian_light, crosswalk e traffic_cone

In [3]:
!cat config/sources.yaml


# Define quais fontes alimentam cada classe nova.
#
# Cada classe (exceto baseline_coco) tem:
#   target_class_id: id global unificado (ver config/classes.yaml)
#   sources: lista de 1 OU MAIS fontes. Cada item tem seu proprio "adapter" e
#            parametros especificos daquele adapter. Todas as fontes de uma
#            mesma classe sao convertidas e remapeadas para o MESMO target_class_id,
#            e depois unificadas (o dedup em src/dedup.py roda no conjunto todo,
#            entao duplicatas entre fontes diferentes da mesma classe sao removidas
#            automaticamente).
#
# Exemplo de classe com 2 fontes: veja "pothole" abaixo (RDD2022 + Roboflow).

baseline_coco:
  adapter: coco_baseline
  images_per_class: 100
  # usa a mesma lista COCO_CLASSES de config/classes.yaml automaticamente

traffic_sign:
  target_class_id: 80
  sources:
    - adapter: tt100k
      # TT100K tem 221 subclasses de placas; por padrao TODAS sao unificadas na
      # classe "traffic_sign". Se q

## 4. Montar o dataset (download + conversão + dedup + split)

In [4]:
!python pipeline.py --steps dataset


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'

Extracting annotations to '/root/fiftyone/coco-2017/raw/instances_train2017.json'

Writing annotations for 8000 downloaded samples to '/root/fiftyone/coco-2017/train/labels.json'
Dataset info written to '/root/fiftyone/coco-2017/info.json'
Loading 'coco-2017' split 'train'

Dataset 'coco-2017-train-8000' created

[coco_baseline:default] 8000 amostras convertidas para YOLO

WARNING ⚠️ Dataset 'TT100K.yaml' images not found, missing path '/content/yolo-traffic-pipeline/datasets/TT100K/images/val'
####################################################

## 5. Treinar

In [5]:
!python pipeline.py --steps train --epochs 30 --imgsz 640 --batch 16 --freeze 10


Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo-traffic-pipeline/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_train, nbs=64, nms=False, opset=None, optimize=False, optimi

## 6. Avaliar por classe

In [9]:
!python pipeline.py --steps evaluate --weights runs/detect/runs/detect/traffic_train/weights/best.pt

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,196,859 parameters, 0 gradients, 8.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1998.8±176.2 MB/s, size: 151.2 KB)
val: Scanning /content/yolo-traffic-pipeline/dataset/labels/val.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 251.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 5.8it/s 16.3s
                   all       1500       9081      0.642      0.478      0.518      0.362
                person        669       2661      0.788      0.629      0.717      0.488
               bicycle         32         59      0.583       0.38      0.404      0.236
                   car        117        409      0.579      0.349      0.393      0.236
            motorcycle         43         81      0.544       0.63      0.643      0.414
              airplane 

In [10]:
import pandas as pd

df = pd.read_csv("reports/per_class_metrics.csv")
df


,class_id,class_name,precision,recall,mAP50,mAP50-95
0,0,person,0.7882,0.6293,0.7168,0.4877
1,1,bicycle,0.5832,0.3796,0.4044,0.2357
2,2,car,0.5787,0.3493,0.3926,0.2363
3,3,motorcycle,0.5442,0.6296,0.6430,0.4140
4,4,airplane,0.8639,0.6800,0.7973,0.5514
...,...,...,...,...,...,...
80,80,traffic_sign,0.6399,0.4671,0.4990,0.3205
81,81,pedestrian_light,0.7154,0.6034,0.6747,0.4204
82,82,crosswalk,0.6564,0.5128,0.6287,0.3569
83,83,traffic_cone,0.7018,0.7633,0.7699,0.4407


## 7. Salvar pesos e relatório no Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs("/content/drive/MyDrive/yolo_traffic_pipeline_outputs", exist_ok=True)
shutil.copy("runs/detect/runs/detect/traffic_train/weights/best.pt",
            "/content/drive/MyDrive/yolo_traffic_pipeline_outputs/best.pt")
shutil.copy("reports/per_class_metrics.csv",
            "/content/drive/MyDrive/yolo_traffic_pipeline_outputs/per_class_metrics.csv")
print("Salvo no Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Salvo no Drive.
